# Collect a small training dataset

Run random steps in two FrozenLake variants using **mouse-env**, save each environment into its own `Datastore`, and push each store to a separate Hugging Face dataset config.

These datasets are intentionally discrete-action only so `02_train_offline.ipynb` can load one config and train the DQN example directly.


In [1]:
from mouse_envs import EnvConfig, make_vector_env
from mouse_core.data import Datastore, push_stores_to_hub

# Hub dataset repo name. A bare name is created under your authenticated HF user.
DATASET_ID = "mouse-example-dataset"

# Number of random environment transitions to collect and upload per config.
STEPS_PER_ENV = 200


## The environments

Each `EnvConfig` has a name. We copy those names onto the matching `Datastore`s so `load_stores_from_hub(..., split=...)` can discover and load them later.


In [ ]:
# FrozenLake has four discrete actions, matching the DQN head in the training notebook.
env_configs = [
    EnvConfig(
        "FrozenLake-v1",
        name="frozenlake_slippery",
        seed=0,
        num_envs=1,
        max_episode_steps=100,
        kwargs={"is_slippery": True},
    ),
    EnvConfig(
        "FrozenLake-v1",
        name="frozenlake_deterministic",
        seed=0,
        num_envs=1,
        max_episode_steps=100,
        kwargs={"is_slippery": False},
    ),
]

## Collect a few steps

Call `step(inputs)`. Get back outputs.
Attach the input (action) we just used, put it first, and append the row.

We do not write an environment column into each row; the dataset config already identifies which environment produced that store.

In [ ]:
def collect_steps(store, env, num_steps):
    """Run random steps and save one flat row per step."""
    for step in range(num_steps):
        inputs = env.sample_random_inputs()
        outputs, _ = env.step(inputs)
        store.append({**inputs[0], **outputs[0]})
        print(f"  step={step}")


stores = []
for cfg in env_configs:
    env = make_vector_env(cfg)
    try:
        store = Datastore(name=cfg.name)
        collect_steps(store, env, STEPS_PER_ENV)
        stores.append(store)
    finally:
        env.close()
    print(store)

frozenlake_slippery inputs={'action': tensor(2)} outputs={'time': tensor(0), 'reward': tensor(0.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.0, 'observation': tensor(0)}
frozenlake_slippery inputs={'action': tensor(1)} outputs={'time': tensor(1), 'reward': tensor(0.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.0, 'observation': tensor(1)}
frozenlake_slippery inputs={'action': tensor(1)} outputs={'time': tensor(2), 'reward': tensor(0.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.0, 'observation': tensor(2)}
frozenlake_slippery inputs={'action': tensor(1)} outputs={'time': tensor(3), 'reward': tensor(0.), 'done': tensor(0), 'episode_index': 0, 'reward_episodic': 0.0, 'observation': tensor(1)}
frozenlake_slippery inputs={'action': tensor(1)} outputs={'time': tensor(4), 'reward': tensor(0.), 'done': tensor(1), 'episode_index': 0, 'reward_episodic': 0.0, 'observation': tensor(5)}
frozenlake_slippery inputs={'action': tensor(2)} outputs={'t

## Push to the Hugging Face Hub

Push each collected store under its matching dataset config name. The Hub client creates or updates the dataset under your authenticated Hugging Face account.


In [4]:
# Each named store becomes one config inside the dataset repo.
url = push_stores_to_hub(
    stores,
    repo_id=DATASET_ID,
    split="train",
    private=False,
    clear=True,
)
print(f"Pushed dataset to {url}")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed to micahr234/mouse-example-dataset (frozenlake_slippery/train: 200, frozenlake_deterministic/train: 200 steps)
Pushed dataset to https://huggingface.co/datasets/micahr234/mouse-example-dataset
